In [0]:
from pyspark.sql import functions as F

# Read the silver table
silver_df = spark.table("students_data.team1_taxi.silver_taxi")

# Compute derived trip metrics
silver_enriched = (
    silver_df
    .withColumn(
        "trip_duration_minutes",
        (F.unix_timestamp("completed_ts") - F.unix_timestamp("time_picked_up_ts")) / 60
    )
    .withColumn(
        "expected_vs_actual_pickup_minutes_difference",
        (F.unix_timestamp("time_picked_up_ts") - F.unix_timestamp("pickup_due_ts")) / 60
    )
    .withColumn(
        "dispatch_to_arrival_minutes",
        (F.unix_timestamp("time_vehicle_arrived_ts") - F.unix_timestamp("time_dispatched_ts")) / 60
    )
    .withColumn("trip_date", F.to_date("pickup_due_ts"))
)

silver_enriched.limit(5).display()

In [0]:
# Build DIM_DATE
date_df = silver_enriched.select("trip_date").distinct().filter(F.col("trip_date").isNotNull())

dim_date = (
    date_df
    .withColumn("date_key", F.date_format("trip_date", "yyyyMMdd").cast("int"))
    .withColumn("full_date", F.col("trip_date"))
    .withColumn("day", F.dayofmonth("trip_date"))
    .withColumn("day_name", F.date_format("trip_date", "EEEE"))
    .withColumn("week", F.weekofyear("trip_date"))
    .withColumn("month", F.month("trip_date"))
    .withColumn("month_name", F.date_format("trip_date", "MMMM"))
    .withColumn("quarter", F.quarter("trip_date"))
    .withColumn("year", F.year("trip_date"))
    .withColumn("weekend_flag", F.dayofweek("trip_date").isin(1, 7))
    .drop("trip_date")
)

weather_daily = spark.table("students_data.team1_taxi.weather_daily")
spark.sql("DROP TABLE IF EXISTS students_data.team1_taxi.dim_date")

dim_date_weather = (
    dim_date
    .join(weather_daily, dim_date["full_date"] == weather_daily["date"], "left")
    .drop("date") 
)

dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("students_data.team1_taxi.dim_date")
print(f"dim_date: {dim_date.count()} rows")
dim_date.display()

In [0]:
# Build DIM_ZONE - Remove duplicates
pickup_zones = silver_enriched.select(
    F.col("pickup_zone").alias("zone_code"),
    F.col("pickup_latitude").alias("latitude"),
    F.col("pickup_longitude").alias("longitude")
)

dest_zones = silver_enriched.select(
    F.col("destination_zone").alias("zone_code"),
    F.col("destination_latitude").alias("latitude"),
    F.col("destination_longitude").alias("longitude")
)

all_zones = pickup_zones.union(dest_zones).filter(F.col("zone_code").isNotNull()).distinct()

# Deduplicate by zone_code, taking first lat/lon
from pyspark.sql.window import Window
w = Window.partitionBy("zone_code").orderBy("latitude")

dim_zone = (
    all_zones
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumn("zone_key", F.monotonically_increasing_id().cast("int"))
)

dim_zone.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("students_data.team1_taxi.dim_zone")
print(f"dim_zone: {dim_zone.count()} rows")
dim_zone.limit(10).display()

In [0]:
# Load dimension keys
dim_date_keys = spark.table("students_data.team1_taxi.dim_date").select("date_key", "full_date")
dim_zone_keys = spark.table("students_data.team1_taxi.dim_zone").select("zone_key", "zone_code")

# Build the fact table by joining dimensions and assigning trip_booked_for_date_key
fact_trip = (
    silver_enriched
    .join(dim_date_keys, silver_enriched["trip_date"] == dim_date_keys["full_date"], "left")
    .withColumn("trip_booked_for_date_key", F.col("date_key"))
    .drop("date_key")
    .join(
        dim_zone_keys.withColumnRenamed("zone_key", "pickup_zone_key"),
        silver_enriched["pickup_zone"] == F.col("zone_code"), "left"
    )
    .drop("zone_code")
    .join(
        dim_zone_keys.withColumnRenamed("zone_key", "destination_zone_key"),
        silver_enriched["destination_zone"] == F.col("zone_code"), "left"
    )
    .drop("zone_code", "full_date")
    .withColumn("trip_key", F.monotonically_increasing_id())
    .select(
        "trip_key", "booking_id", "trip_booked_for_date_key",
        "pickup_zone_key", "destination_zone_key",
        F.col("trip_status").alias("status"),
        "payment_type", "booking_source",
        F.col("driver").alias("driver_id"),
        F.col("vehicle").alias("vehicle_id"),
        F.col("priority").alias("priority_level"),
        F.col("capabilities").alias("capabilities"),
        F.col("booked_by").alias("dispatcher_id"),
        F.col("price").alias("fare_amount"),
        F.col("time_dispatched_ts").alias("dispatch_timestamp"),
        F.col("pickup_due_ts").alias("pickup_due_timestamp"),
        "distance",
        "expected_vs_actual_pickup_minutes_difference",
        "dispatch_to_arrival_minutes",
        "trip_duration_minutes"
    )
)

fact_trip.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("students_data.team1_taxi.fact_trip")
print(f"fact_trip: {fact_trip.count()} rows")
fact_trip.limit(5).display()

In [0]:
# Gold aggregate: daily KPIs
gold_daily_kpis = (
    spark.table("students_data.team1_taxi.fact_trip")
    .join(spark.table("students_data.team1_taxi.dim_date"), F.col("trip_booked_for_date_key") == F.col("date_key"), "inner")
    .groupBy("full_date", "day_name")
    .agg(
        F.sum("fare_amount").alias("total_revenue"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_trip_duration_min"),
        F.count("trip_key").alias("total_trips"),
        F.round(F.avg("distance"), 2).alias("avg_distance_km")
    )
    .orderBy("full_date")
)

gold_daily_kpis.write.format("delta").mode("overwrite").saveAsTable("students_data.team1_taxi.gold_daily_kpis")
gold_daily_kpis.display()